In [18]:
# --------------------------------------------------
# 1. Imports and paths
# --------------------------------------------------
from pathlib import Path
import subprocess

# Paths
ogs_bin = Path(r"C:\Users\dominic.becerra\Documents\OGS\ogs\bin")
input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")
mesh_msh = Path(r"E:\ADATA\LITHIUM\OGS\Jupyter notebooks\mesh5.msh")

# Tools
gmsh2ogs = ogs_bin / "GMSH2OGS.exe"

# Output base names
bulk_vtu = input_dir / "mesh5.vtu"
boundary_prefix = input_dir / "mesh5_boundary_"

In [19]:
# --------------------------------------------------
# 2. Generate bulk mesh + boundary surfaces
# --------------------------------------------------
result = subprocess.run(
    [
        str(gmsh2ogs),
        "-i", str(mesh_msh),
        "-o", str(bulk_vtu),
        "-b",                       # generate boundary surfaces
        "-e",                       # exclude lines (optional)
        "--gmsh2_physical_id"       # convert physical ids correctly
    ],
    cwd=input_dir,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

info: Reading E:\ADATA\LITHIUM\OGS\Jupyter notebooks\mesh5.msh.
info: 	... finished.
info: Nr. Nodes: 28127.
info: Nr. Elements: 167161.
info: Time for reading: 2.442681 seconds.
info: Read 28127 nodes and 167161 elements.
info: No elements to remove
info: Mesh does not contain any lines.
info: Please check your mesh carefully!
info: Degenerated or redundant mesh elements can cause OGS to stop or misbehave.
info: Use the -e option to delete redundant line elements.
info: Axis aligned bounding box: 	x [0, 10) (extent 10)
	y [0, 10) (extent 10)
	z [0, 5) (extent 5)
info: Edge length: [0.123624, 0.630171]
info: Number of elements in the mesh:
info: 	Triangles: 12720
info: 	Tetrahedrons: 154441
info: There are 1 properties in the mesh:
info: 	MaterialIDs: (167161 values) [0, 30]




In [20]:
import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

bulk = meshio.read(input_dir / "mesh5.vtu")

print("Cell data keys:", bulk.cell_data_dict.keys())

for key in bulk.cell_data_dict.keys():
    print("\nKEY:", key)
    for ctype, data in bulk.cell_data_dict[key].items():
        print(ctype, np.unique(data)[:20])

Cell data keys: dict_keys(['MaterialIDs'])

KEY: MaterialIDs
triangle [ 0  1  2  3  5  8 10 12 13 14 15 16 17 18 19 20 21 22 23 24]
tetra [ 3  4  5  6  7  8  9 10 11 12 13]


In [21]:
# --------------------------------------------------
# 3. Generate well submeshes (volumes)
# --------------------------------------------------

import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

bulk = meshio.read(input_dir / "mesh5.vtu")

tetra = bulk.cells_dict["tetra"]
mat_tetra = bulk.cell_data_dict["MaterialIDs"]["tetra"]

# Map new material IDs for wells
well_tags = {
    3: "well1",  # injection
    4: "well2"   # production
}

for tag, name in well_tags.items():
    selected = tetra[mat_tetra == tag]

    used_points = np.unique(selected.flatten())
    old_to_new = {old: new for new, old in enumerate(used_points)}
    new_points = bulk.points[used_points]

    new_tetra = np.array(
        [[old_to_new[node] for node in elem] for elem in selected],
        dtype=int
    )

    bulk_node_ids = used_points.astype(np.uint64)

    submesh = meshio.Mesh(
        points=new_points,
        cells=[("tetra", new_tetra)],
        point_data={"bulk_node_ids": bulk_node_ids},
        cell_data={"MaterialIDs": [np.full(len(new_tetra), tag, dtype=np.int32)]}
    )

    out = input_dir / f"{name}.vtu"
    meshio.write(out, submesh)
    print(f"Created {out.name}, tetra cells: {len(new_tetra)}, points: {len(new_points)}, bulk_node_ids dtype: {bulk_node_ids.dtype}")

Created well1.vtu, tetra cells: 19045, points: 5574, bulk_node_ids dtype: uint64
Created well2.vtu, tetra cells: 27144, points: 6139, bulk_node_ids dtype: uint64


In [22]:
from pathlib import Path
import subprocess

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")
ogs_bin = Path(r"C:\Users\dominic.becerra\Documents\OGS\ogs\bin")

identify = ogs_bin / "identifySubdomains.exe"

for fname in ["well1.vtu", "well2.vtu"]:
    result = subprocess.run(
        [
            str(identify),
            "-m", str(input_dir / "mesh5.vtu"),   # bulk mesh
            "-f", str(input_dir / fname)          # submesh
        ],
        cwd=input_dir,
        capture_output=True,
        text=True
    )
    print(fname)
    print(result.stdout)
    print(result.stderr)

well1.vtu
info: Mesh reading time: 0.248989 s
info: MeshNodeSearcher construction time: 0.0050469 s
info: identifySubdomainMesh(): identifySubdomainMeshNodes took 0.0018681 s
info: There is already a 'bulk_node_ids' property present in the subdomain mesh 'well1' and it is equal to the newly computed values.
info: identifySubdomainMesh(): identifySubdomainMeshElements took 0.212951 s
info: identifySubdomains time: 0.216498 s
info: writing time: 0.117171 s
info: Entire run time: 0.587997 s


well2.vtu
info: Mesh reading time: 0.263874 s
info: MeshNodeSearcher construction time: 0.0055509 s
info: identifySubdomainMesh(): identifySubdomainMeshNodes took 0.0020581 s
info: There is already a 'bulk_node_ids' property present in the subdomain mesh 'well2' and it is equal to the newly computed values.
info: identifySubdomainMesh(): identifySubdomainMeshElements took 0.257097 s
info: identifySubdomains time: 0.260883 s
info: writing time: 0.121025 s
info: Entire run time: 0.651628 s




In [23]:
import meshio
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

for fname in ["well1.vtu", "well2.vtu"]:
    m = meshio.read(input_dir / fname)
    print("\n", fname)
    print("Number of points:", len(m.points))
    print("Cell blocks:", m.cells)
    print("Cell types:", list(m.cells_dict.keys()))
    print("Point data:", m.point_data.keys())
    print("Cell data:", m.cell_data_dict.keys())


 well1.vtu
Number of points: 5574
Cell blocks: [<meshio CellBlock, type: tetra, num cells: 19045, tags: []>]
Cell types: ['tetra']
Point data: dict_keys(['bulk_node_ids'])
Cell data: dict_keys(['MaterialIDs', 'bulk_element_ids'])

 well2.vtu
Number of points: 6139
Cell blocks: [<meshio CellBlock, type: tetra, num cells: 27144, tags: []>]
Cell types: ['tetra']
Point data: dict_keys(['bulk_node_ids'])
Cell data: dict_keys(['MaterialIDs', 'bulk_element_ids'])


In [24]:
import meshio
import numpy as np
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

m = meshio.read(input_dir / "mesh5.vtu")

mat = m.cell_data_dict["MaterialIDs"]["tetra"]

print("Unique MaterialIDs in tetra:")
print(np.unique(mat, return_counts=True))

Unique MaterialIDs in tetra:
(array([ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13], dtype=int32), array([19045, 27144, 27269,    81,    75, 18926,    47,    51, 61669,
          66,    68]))


In [10]:
import meshio
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

m = meshio.read(input_dir / "mesh5.vtu")

print("Cells in mesh:", m.cells)
print("Cell data keys:", m.cell_data_dict.keys())
for key in m.cell_data_dict.keys():
    for ctype, data in m.cell_data_dict[key].items():
        print(f"Cell type: {ctype}, data length: {len(data)}, unique values: {np.unique(data)[:10]}")

Cells in mesh: [<meshio CellBlock, type: triangle, num cells: 12720, tags: []>, <meshio CellBlock, type: tetra, num cells: 154441, tags: []>]
Cell data keys: dict_keys(['gmsh:physical', 'gmsh:geometrical'])
Cell type: triangle, data length: 12720, unique values: [301 302 303 304 305 306 401 402 403 404]
Cell type: tetra, data length: 154441, unique values: [101 102 103 104 105 201 202]
Cell type: triangle, data length: 12720, unique values: [ 1  2  3  4  6  9 11 13 14 15]
Cell type: tetra, data length: 154441, unique values: [ 4  5  6  7  8  9 10 11 12 13]
